# 10 — IFRS Taxonomy Builder

Builds and maintains the IFRS taxonomy used by Numera for financial
statement mapping. Parses official IFRS taxonomy XBRL schemas,
generates synonym lists, and exports a structured JSON taxonomy
for the ML service.

**Output**: `ifrs_taxonomy.json` with all elements, labels, synonyms

In [ ]:
# Cell 1: Setup
!pip install -q lxml tqdm PyYAML

import json, re
from pathlib import Path
from lxml import etree
from tqdm.auto import tqdm
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

DATA_DIR = Path(cfg['drive']['data_dir'])
OUTPUT_DIR = Path('../../data')
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = DATA_DIR

print('Ready ✅')

In [ ]:
# Cell 2: Define IFRS taxonomy elements
# Core IFRS elements for financial statements

IFRS_ELEMENTS = {
    # ── Income Statement ──
    'Revenue': {'category': 'income_statement', 'balance': 'credit',
                'synonyms': ['Sales', 'Turnover', 'Net Revenue', 'Total Revenue', 'Net Sales']},
    'CostOfSales': {'category': 'income_statement', 'balance': 'debit',
                    'synonyms': ['Cost of Revenue', 'Cost of Goods Sold', 'COGS', 'Cost of Sales']},
    'GrossProfit': {'category': 'income_statement', 'balance': 'credit',
                    'synonyms': ['Gross Margin', 'Gross Income']},
    'SellingGeneralAdministrativeExpenses': {'category': 'income_statement', 'balance': 'debit',
                                             'synonyms': ['SG&A', 'Selling and Admin', 'Operating Expenses']},
    'OperatingProfit': {'category': 'income_statement', 'balance': 'credit',
                        'synonyms': ['Operating Income', 'EBIT', 'Operating Earnings']},
    'FinanceCosts': {'category': 'income_statement', 'balance': 'debit',
                     'synonyms': ['Interest Expense', 'Finance Charges', 'Interest Costs']},
    'FinanceIncome': {'category': 'income_statement', 'balance': 'credit',
                      'synonyms': ['Interest Income', 'Finance Revenue']},
    'ProfitBeforeTax': {'category': 'income_statement', 'balance': 'credit',
                        'synonyms': ['Income Before Tax', 'Pre-tax Income', 'EBT']},
    'IncomeTaxExpense': {'category': 'income_statement', 'balance': 'debit',
                         'synonyms': ['Tax Expense', 'Provision for Taxes', 'Income Taxes']},
    'ProfitForThePeriod': {'category': 'income_statement', 'balance': 'credit',
                           'synonyms': ['Net Income', 'Net Profit', 'Net Earnings', 'Profit After Tax']},
    'BasicEarningsPerShare': {'category': 'income_statement', 'balance': 'credit',
                              'synonyms': ['Basic EPS', 'Earnings Per Share']},
    'DilutedEarningsPerShare': {'category': 'income_statement', 'balance': 'credit',
                                'synonyms': ['Diluted EPS']},
    'DepreciationAndAmortisation': {'category': 'income_statement', 'balance': 'debit',
                                    'synonyms': ['D&A', 'Depreciation', 'Amortization']},

    # ── Balance Sheet — Assets ──
    'CashAndCashEquivalents': {'category': 'balance_sheet', 'balance': 'debit',
                               'synonyms': ['Cash', 'Cash & Equivalents', 'Cash and Bank']},
    'TradeAndOtherReceivables': {'category': 'balance_sheet', 'balance': 'debit',
                                 'synonyms': ['Accounts Receivable', 'Trade Receivables', 'Receivables']},
    'Inventories': {'category': 'balance_sheet', 'balance': 'debit',
                    'synonyms': ['Inventory', 'Stock', 'Merchandise']},
    'CurrentAssets': {'category': 'balance_sheet', 'balance': 'debit',
                      'synonyms': ['Total Current Assets']},
    'PropertyPlantAndEquipment': {'category': 'balance_sheet', 'balance': 'debit',
                                  'synonyms': ['PP&E', 'Fixed Assets', 'Property and Equipment']},
    'IntangibleAssets': {'category': 'balance_sheet', 'balance': 'debit',
                         'synonyms': ['Intangibles', 'Goodwill and Intangibles']},
    'Goodwill': {'category': 'balance_sheet', 'balance': 'debit',
                 'synonyms': ['Goodwill on Acquisition']},
    'NoncurrentAssets': {'category': 'balance_sheet', 'balance': 'debit',
                         'synonyms': ['Non-current Assets', 'Long-term Assets', 'Fixed Assets']},
    'TotalAssets': {'category': 'balance_sheet', 'balance': 'debit',
                    'synonyms': ['Total Assets']},

    # ── Balance Sheet — Liabilities ──
    'TradeAndOtherPayables': {'category': 'balance_sheet', 'balance': 'credit',
                              'synonyms': ['Accounts Payable', 'Trade Payables', 'Payables']},
    'ShortTermBorrowings': {'category': 'balance_sheet', 'balance': 'credit',
                            'synonyms': ['Current Borrowings', 'Short-term Debt', 'Current Debt']},
    'CurrentLiabilities': {'category': 'balance_sheet', 'balance': 'credit',
                            'synonyms': ['Total Current Liabilities']},
    'LongTermBorrowings': {'category': 'balance_sheet', 'balance': 'credit',
                           'synonyms': ['Long-term Debt', 'Non-current Borrowings']},
    'NoncurrentLiabilities': {'category': 'balance_sheet', 'balance': 'credit',
                              'synonyms': ['Non-current Liabilities', 'Long-term Liabilities']},
    'TotalLiabilities': {'category': 'balance_sheet', 'balance': 'credit',
                         'synonyms': ['Total Liabilities']},
    'TotalEquity': {'category': 'balance_sheet', 'balance': 'credit',
                    'synonyms': ['Shareholders Equity', 'Stockholders Equity', 'Net Assets']},
    'IssuedCapital': {'category': 'balance_sheet', 'balance': 'credit',
                      'synonyms': ['Share Capital', 'Common Stock', 'Ordinary Shares']},
    'RetainedEarnings': {'category': 'balance_sheet', 'balance': 'credit',
                         'synonyms': ['Accumulated Profit', 'Retained Profit']},

    # ── Cash Flow Statement ──
    'CashFlowsFromOperatingActivities': {'category': 'cash_flow', 'balance': 'debit',
                                          'synonyms': ['Operating Cash Flow', 'CFO', 'Cash from Operations']},
    'CashFlowsFromInvestingActivities': {'category': 'cash_flow', 'balance': 'debit',
                                          'synonyms': ['Investing Cash Flow', 'CFI', 'Cash from Investing']},
    'CashFlowsFromFinancingActivities': {'category': 'cash_flow', 'balance': 'debit',
                                          'synonyms': ['Financing Cash Flow', 'CFF', 'Cash from Financing']},
    'CapitalExpenditures': {'category': 'cash_flow', 'balance': 'debit',
                            'synonyms': ['CapEx', 'Capital Spending', 'Purchases of PP&E']},
    'DividendsPaid': {'category': 'cash_flow', 'balance': 'debit',
                      'synonyms': ['Dividends', 'Cash Dividends Paid']},
}

print(f'Defined {len(IFRS_ELEMENTS)} taxonomy elements')

In [ ]:
# Cell 3: Convert CamelCase element names to human-readable labels

def camel_to_label(name: str) -> str:
    """Convert CamelCase to space-separated words."""
    result = re.sub(r'([A-Z])', r' \1', name).strip()
    return result

taxonomy = []
for code, info in IFRS_ELEMENTS.items():
    label = camel_to_label(code)
    taxonomy.append({
        'code': f'ifrs-full:{code}',
        'name': code,
        'label': label,
        'category': info['category'],
        'balance': info['balance'],
        'synonyms': info['synonyms'],
    })

# Sort by category then name
taxonomy.sort(key=lambda x: (x['category'], x['name']))
print(f'Built taxonomy with {len(taxonomy)} elements')
for t in taxonomy[:5]:
    print(f"  {t['code']}: {t['label']} ({len(t['synonyms'])} synonyms)")

In [ ]:
# Cell 4: Generate additional synonym variations

def generate_variations(label: str, synonyms: list[str]) -> list[str]:
    """Generate common text variations for better matching."""
    variations = set(synonyms)
    variations.add(label)
    variations.add(label.lower())
    variations.add(label.upper())
    # Add "Total" prefix/suffix variants
    if not label.startswith('Total'):
        variations.add(f'Total {label}')
    # Add "Net" variants for relevant items
    if 'income' in label.lower() or 'profit' in label.lower():
        variations.add(f'Net {label}')
    return sorted(variations)

for item in taxonomy:
    item['synonyms'] = generate_variations(item['label'], item['synonyms'])

total_synonyms = sum(len(t['synonyms']) for t in taxonomy)
print(f'Total synonym entries: {total_synonyms}')

In [ ]:
# Cell 5: Export taxonomy JSON
output = {
    'version': '2024.1',
    'framework': 'IFRS',
    'element_count': len(taxonomy),
    'categories': sorted(set(t['category'] for t in taxonomy)),
    'elements': taxonomy,
}

out_file = OUTPUT_DIR / 'ifrs_taxonomy.json'
with open(out_file, 'w') as f:
    json.dump(output, f, indent=2)

# Also save to data dir for training notebooks
data_copy = DATA_DIR / 'ifrs_taxonomy.json'
with open(data_copy, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Taxonomy saved:')
print(f'  {out_file}')
print(f'  {data_copy}')
print(f'\nCategories:')
from collections import Counter
for cat, count in Counter(t['category'] for t in taxonomy).most_common():
    print(f'  {cat}: {count} elements')